<a href="https://colab.research.google.com/github/crezny/DATASCI266_final_project/blob/main/Baseline_Scoring_local.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%%capture
!pip install -q -U transformers
!pip install -q -U accelerate
!pip install -q -U bitsandbytes
!pip install -q -U evaluate
!pip install -q rouge_score

In [4]:
import pandas as pd
import evaluate
import re
import os
import torch
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer, scoring
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Optional, Tuple, Any, Sequence
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import math
import matplotlib.pyplot as plt
from transformers import BitsAndBytesConfig

### LLM SSM Evaluation

In [5]:
import torch

"""Load  semantic similarity model"""


#model_name='BAAI/bge-large-en-v1.5'
sentence_model_name = 'BAAI/bge-large-en-v1.5'
#alternate_sentence_transformer = 'all-MiniLM-L6-v2'
print(f"Loading model: {sentence_model_name}...")
sentence_model = SentenceTransformer(sentence_model_name,
                                      device='cuda',  # Must be on GPU for bitsandbytes
                                      model_kwargs={
                                          'load_in_8bit': True,
                                          'torch_dtype': torch.float16
                                      })

print("Model loaded successfully!\n")


# Load Rouge Scorer
rouge = evaluate.load('rouge')
bleu = evaluate.load("bleu")

bleurt_tokenizer = AutoTokenizer.from_pretrained("Elron/bleurt-base-512")
bleurt_model = AutoModelForSequenceClassification.from_pretrained("Elron/bleurt-base-512")

Loading model: BAAI/bge-large-en-v1.5...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [6]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')


# Define your directory
output_folder = "/content/drive/MyDrive/mids266_project/output"
data_folder = "/content/drive/MyDrive/mids266_project/data"

# Ensure the output directory exists
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

Mounted at /content/drive


In [7]:
# Load input file to get the gold_text
df_gold = pd.read_csv(f'{data_folder}/datasource.csv')
df_gold.head(5)

,policy_id,policy_title,source_framework,policy_origin,section_id,section_title,is_heading,clause_id,clause_text,clause_full_text,clause_section_number,company_type,company_id
0,CL_0001-AUP-v2,Acceptable Use Policy,NIST CSF,client,CL_0001-AUP-v2:S1,PURPOSE,False,CL_0001-AUP-v2:1.context.523a34,The purpose of this policy is to outline the a...,The purpose of this policy is to outline the a...,1,Client,CL_0001
1,CL_0001-AUP-v2,Acceptable Use Policy,NIST CSF,client,CL_0001-AUP-v2:S2,SCOPE,False,CL_0001-AUP-v2:2.context.5af3cd,This policy applies to all City personnel and ...,This policy applies to all City personnel and ...,2,Client,CL_0001
2,CL_0001-AUP-v2,Acceptable Use Policy,NIST CSF,client,CL_0001-AUP-v2:S3.1,General Use and Ownership,False,CL_0001-AUP-v2:3.1.context.bf694e,[COMPANY_NAME] information stored on electroni...,[COMPANY_NAME] information stored on electroni...,3.1,Client,CL_0001
3,CL_0001-AUP-v2,Acceptable Use Policy,NIST CSF,client,CL_0001-AUP-v2:S3.1,General Use and Ownership,False,CL_0001-AUP-v2:3.1.context.efab41,All personnels and contractors have a responsi...,All personnels and contractors have a responsi...,3.1,Client,CL_0001
4,CL_0001-AUP-v2,Acceptable Use Policy,NIST CSF,client,CL_0001-AUP-v2:S3.1,General Use and Ownership,False,CL_0001-AUP-v2:3.1.context.5e59b4,"All personnels and contractors may access, use...","All personnels and contractors may access, use...",3.1,Client,CL_0001


In [8]:
# display counts for section_title grouped by policy_title in descending order

section_title_counts = df_gold.value_counts(['source_framework', 'policy_title', 'section_title'])
#section_title_counts = section_title_counts.sort_values(ascending=False)
section_title_counts[section_title_counts > 40]

source_framework  policy_title                                   section_title                 
ISO 27001         Information Security Management System Plan    PERFORMANCE EVALUATION            81
SOC2              Data Classification and Data Retention Policy  [COMPANY_NAME] Confidential       69
NIST CSF          Document Ownership and Revisions               Content of Audit Records          66
                  Information System Protection Policy           Hardening                         63
                  Data Classification and Data Retention Policy  [COMPANY_NAME] Confidential       60
SOC2              Acceptable Use Policy                          System and Network Activities     58
                  Logging and Monitoring Policy                  General Logging and Monitoring    54
ISO 27001         Information Security Management System Plan    CONTEXT OF THE ORGANIZATION       52
SOC2              Acceptable Use Policy                          Software Usage                    45
                  Logging and Monitoring Policy                  Activities to be Logged           45
NIST CSF          Acceptable Use Policy                          System and Network Activities     45
SOC2              Data Classification and Data Retention Policy  [COMPANY_NAME] Restricted         44
Name: count, dtype: int64

In [9]:
# group by source_framework, policy_title_section_title
policy_text_counts = df_gold.groupby(['source_framework', 'policy_title', 'section_title']).size()
policy_text_counts[policy_text_counts > 20]

source_framework  policy_title                                   section_title                     
ISO 27001         Information Security Management System Plan    CONTEXT OF THE ORGANIZATION           52
                                                                 LEADERSHIP                            38
                                                                 PERFORMANCE EVALUATION                81
                                                                 PLANNING                              33
                                                                 SUPPORT                               25
NIST CSF          Acceptable Use Policy                          Email and Communication Activities    24
                                                                 General unacceptable Use              38
                                                                 Software Usage                        34
                                                                 System and Network Activities         45
                  Asset Management Policy                        Policy                                39
                  Data Classification and Data Retention Policy  ROLES AND RESPONSIBILITIES            22
                                                                 [COMPANY_NAME] Confidential           60
                                                                 [COMPANY_NAME] Internal               32
                                                                 [COMPANY_NAME] Restricted             36
                  Document Ownership and Revisions               Content of Audit Records              66
                                                                 Continuous Monitoring                 30
                  Information System Protection Policy           Hardening                             63
                  Patch Management Policy                        Prioritization of Patches             25
                  Software Development Lifecycle Policy          SDLC Phases and Processes             29
SOC2              Acceptable Use Policy                          E-mail Usage                          23
                                                                 Email and Communication Activities    32
                                                                 General unacceptable Use              39
                                                                 Software Usage                        45
                                                                 System and Network Activities         58
                  Access Control Policy                          Roles and Responsibilities            34
                  Bring Your Own Device Policy                   Policy and Procedures                 35
                  Change Management Policy                       Administration                        24
                                                                 Change Management Details             32
                                                                 General                               31
                  Data Classification and Data Retention Policy  ROLES AND RESPONSIBILITIES            22
                                                                 [COMPANY_NAME] Confidential           69
                                                                 [COMPANY_NAME] Internal               38
                                                                 [COMPANY_NAME] Restricted             44
                  Incident Response Plan                         Declaration                           32
                  Logging and Monitoring Policy                  Activities to be Logged               45
                                                                 Application Log Elements              33
                                                                 General Logging and

In [10]:
# Load the output file
df_output = pd.read_csv(f'{output_folder}/baseline_results.csv')
df_output

,policy_id,clause_text,model_name,output,time_taken,token_count,policy_title,section_number,section_text,section_title,source_framework,variant,model_type
0,CL_0001-ASMP-v1.0,"3.10.2 Mobile Devices: Smartphones, Tablets, E...",Gemma-2-9B,"This policy outlines the acceptable use, secur...",24.73,218,Asset Management Policy,3.10.2,"3.10.2 Mobile Devices: Smartphones, Tablets, E...","Mobile Devices: Smartphones, Tablets, Etc.",NIST CSF,A,baseline
1,CL_0001-ASMP-v1.0,"3.11 System and Device Maintenance\nSystems, d...",Gemma-2-9B,System and device maintenance activities are e...,20.65,183,Asset Management Policy,3.11,"3.11 System and Device Maintenance\nSystems, d...",System and Device Maintenance,NIST CSF,A,baseline
2,CL_0001-ITP-v3.0,1 PURPOSE\nThis document outlines the policies...,Gemma-2-9B,This Information Technology Policy establishes...,22.44,198,Information Technology Policy,1,1 PURPOSE\nThis document outlines the policies...,PURPOSE,NIST CSF,A,baseline
3,CL_0001-ITP-v3.0,2 SCOPE\nThis policy applies to all City perso...,Gemma-2-9B,This Information Technology (IT) Policy establ...,23.92,213,Information Technology Policy,2,2 SCOPE\nThis policy applies to all City perso...,SCOPE,NIST CSF,A,baseline
4,CL_0001-ITP-v3.0,3.2.2 Password Protection\nPasswords must be t...,Gemma-2-9B,"All users are required to utilize strong, uniq...",15.27,134,Information Technology Policy,3.2.2,3.2.2 Password Protection\nPasswords must be t...,Password Protection,NIST CSF,A,baseline
...,...,...,...,...,...,...,...,...,...,...,...,...,...
255,SOC_2-AMP-v1.0,3.6 Privileged Access Management\nProcesses sh...,SmolLM3,"Okay, I need to draft the ""Privileged Access M...",18.98,254,Access Management Policy,3.6,3.6 Privileged Access Management\nProcesses sh...,Privileged Access Management,SOC2,A,baseline
256,SOC_2-AUP-v1.0,3.6 Security and Proprietary Information\nAll ...,SmolLM3,"Okay, I need to draft the ""Security and Propri...",19.15,254,Acceptable Use Policy,3.6,3.6 Security and Proprietary Information\nAll ...,Security and Proprietary Information,SOC2,A,baseline
257,SOC_2-LMP-v1.0,2 SCOPE\nThis policy applies to all [COMPANY_N...,SmolLM3,"Okay, I need to draft the ""Scope (Section 2)"" ...",18.93,254,Logging and Monitoring Policy,2,2 SCOPE\nThis policy applies to all [COMPANY_N...,SCOPE,SOC2,A,baseline
258,SOC_2-LMP-v1.0,3.3 Underlying Requirements\nAll systems that ...,SmolLM3,"Okay, I need to draft the ""Underlying Requirem...",19.14,254,Logging and Monitoring Policy,3.3,3.3 Underlying Requirements\nAll systems that ...,Underlying Requirements,SOC2,A,baseline


In [11]:

output_text_counts = df_output.groupby([ 'source_framework','policy_title', 'section_number']).size()
output_text_counts


source_framework  policy_title                                   section_number
ISO 27001         Cryptography and Encryption Policy             5                  4
                  Information Security Management System Plan    8                  4
                  Personnel Security Policy                      2                  4
                                                                 3.3                4
                  Physical Security Policy                       3.1                4
                  Security Program Compliance Policy             4                  4
NIST CSF          Access Control Policy                          3.8                4
                  Asset Management Policy                        3.10.2             4
                                                                 3.11               4
                  Backup Policy                                  4.7                4
                  Backup and Data Retention Policy               2                  4
                                                                 5                  4
                  Business Continuity & Disaster Recovery Plan   9.1                4
                                                                 9.2.2              4
                                                                 9.2.7              4
                  Change Management Policy                       3.5                4
                  Cybersecurity Awareness Training Policy        3.4                8
                  Data Classification and Data Retention Policy  5.1.5              4
                  Information Security Policy                    3.10               4
                  Information System Protection Policy           4                  4
                  Information Technology Policy                  1                  4
                                                                 2                  4
                                                                 3.2.2              4
                                                                 3.3                4
                                                                 3.4.7              4
                                                                 3.5.1              4
                  Logging and Monitoring Policy                  3.6                4
                  Patch & Vulnerability Management Policy        1                  8
                  Physical & Environmental Security Policy       3.11               8
                                                                 3.2                8
                  System and Device Maintenance Policy           1                  4
                                                                 3.1                4
                  Third Party Management Policy                  5                  8
SOC2              Acceptable Use Policy                          3.6               12
                  Access Control Policy                          3.5.1              4
                  Access Management Policy                       3.6               12
                  Asset Management Policy                        3.2                4
                  Backup Policy                                  3.8                4
                  Bring Your Own Device Policy                   3.11               4
                                                                 3.4                4
                  Business Continuity & Disaster Recovery Plan   10.1.5             4
                  Endpoint Protection Policy                     5                  4
                  Incident & Outage Management Policy            3.3                4
                  Information Security Policy                    1                  4
                  Logging and Monitoring Policy                  2                 12
                                                  

In [13]:
def create_scores(
        filepath: str
):
    df = pd.read_csv(filepath)
    df['time_taken'] = pd.to_timedelta(df['time_taken'])

    df = df.assign(
        output=df["output"].fillna("").astype(str).str.strip().apply(lambda x: x.strip() if not isinstance(x, str) else re.sub(r"<think>.*?</think>", "", x, flags=re.DOTALL).strip()),
        #golden=df["org_clause_text"].fillna("").astype(str).str.strip()
        golden=df_gold["clause_text"].fillna("").astype(str).str.strip()
    )
    preds = df["output"].fillna("").astype(str).tolist()
    refs = df["golden"].fillna("").astype(str).tolist()

    E_out = embed(preds)
    E_gold = embed(refs)


    rouge_scores = rouge.compute(predictions=preds, references=refs, use_aggregator=False)
    bleu_results = bleu.compute(predictions=preds, references=refs)


    time_list = df['time_taken'].dt.total_seconds().tolist()
    # Normalize to unit vectors for cosine via dot product
    eps = 1e-12
    E_out  = E_out  / (np.linalg.norm(E_out,  axis=1, keepdims=True) + eps)
    E_gold = E_gold / (np.linalg.norm(E_gold, axis=1, keepdims=True) + eps)
    sts_scores = np.einsum('ij,ij->i', E_out, E_gold)
    return sts_scores, time_list, rouge_scores, bleu_results

#model = SentenceTransformer("BAAI/bge-large-en-v1.5")
model = sentence_model

def embed(texts, batch_size=64):
    # Recommended BGE instruction-style prefix; use the SAME for both sides
    prefixed = [f"Represent this sentence for measuring semantic similarity: {t}" for t in texts]
    return model.encode(
        prefixed,
        batch_size=batch_size,
        normalize_embeddings=True,        # important: makes dot product == cosine
        convert_to_numpy=True,
        show_progress_bar=True
    )

In [14]:
def calculate_mds(df, sts_scores, rouge_scores):
  """ Calculate MDS score = sts_scores * rougeLsum
  """
  epsilon = 1e-8

  # Convert rougeLsum to numpy array
  rougeLsum_scores = np.array(rouge_scores['rougeLsum'])

  #  # Calculate keyword overlap score
    #TODO

  # Calculate length ratio (symmetric, ranges [0, 1])
  len_pred = np.array([len(p.split()) for p in df["output"]])  # word count
  len_ref = np.array([len(r.split()) for r in df["section_text"]])

  length_scores = np.minimum(len_pred, len_ref) / (np.maximum(len_pred, len_ref) + epsilon)


  # Weighted Harmonic Mean (3 terms)
  w1, w2, w3 = 0.4, 0.4, 0.4  # need to adjust


  mds_scores = 1 / (
    w1 / (sts_scores + epsilon) +
    w2 / (rougeLsum_scores + epsilon) +
    w3 / (length_scores + epsilon)
  )


  return mds_scores, length_scores

In [15]:
def create_hierarchical_scores(df, model_name):

    hierarchical_scores = hierarchical = df.groupby(['source_framework'])[
      ['mds_scores', 'rouge_scores', 'length_scores', 'sts_scores']].agg(['median', 'count']).reset_index()

    print(hierarchical_scores)

    # write to csv file
    # Flatten column names
    hierarchical_scores.columns = ['_'.join(col) for col in hierarchical_scores.columns]

    # Reset index and save

    filename = f'hierarchical_scores_{model_name}_{sentence_model_name.replace('/', '-')}.csv'
    hierarchical_scores.reset_index().to_csv(f'{output_folder}/filename', index=False)



    return hierarchical_scores

In [16]:
def create_scores_1(
        output_df,
        model_name,

):
    df = output_df.copy()
    df = df[df["model_name"]== model_name]
    df['time_taken'] = pd.to_timedelta(df['time_taken'])

    df = df.assign(
        output=df["output"].fillna("").astype(str).str.strip().apply(lambda x: x.strip() if not isinstance(x, str) else re.sub(r"<think>.*?</think>", "", x, flags=re.DOTALL).strip()),
        #golden=df["org_clause_text"].fillna("").astype(str).str.strip()
        golden=df["section_text"].fillna("").astype(str).str.strip()
    )
    preds = df["output"].fillna("").astype(str).tolist()
    refs = df["golden"].fillna("").astype(str).tolist()

    E_out = embed(preds)
    E_gold = embed(refs)


    rouge_scores = rouge.compute(predictions=preds, references=refs, use_aggregator=False)
    bleu_results = bleu.compute(predictions=preds, references=refs)


    time_list = df['time_taken'].dt.total_seconds().tolist()
    # Normalize to unit vectors for cosine via dot product
    eps = 1e-12
    E_out  = E_out  / (np.linalg.norm(E_out,  axis=1, keepdims=True) + eps)
    E_gold = E_gold / (np.linalg.norm(E_gold, axis=1, keepdims=True) + eps)
    sts_scores = np.einsum('ij,ij->i', E_out, E_gold)

    # Calcualte MDS score
    mds_scores, length_scores = calculate_mds(df, sts_scores, rouge_scores)

     # Calculate scores by Framework/Policy/Section
    df["mds_scores"] = mds_scores
    df["length_scores"] = length_scores
    df["sts_scores"] = sts_scores
    df["rouge_scores"] = np.array(rouge_scores['rougeLsum'])
    hierarchical_scores = create_hierarchical_scores(df, model_name)


    return mds_scores, length_scores, sts_scores, time_list, rouge_scores, bleu_results



In [17]:
# df has columns: "output", "golden"
# Make sure NaNs are handled
"""
SmolLM3_sts_scores, SmolLM3_time_list, SmolLM3_scores, SmolLM3_bleu_results = create_scores('data/baseline_results/baseline_SmolLM3_outputs.csv')
Llama_3_2_3B_Instruct_sts_scores, Llama_3_2_3B_Instruct_time_list, Llama_3_2_3B_Instruct_scores, Llama_3_2_3B_Instruct_bleu_results = create_scores('data/baseline_results/baseline_llama-3.2-3B-Instruct_outputs.csv')
Gemma_2_9B_sts_scores, Gemma_2_9B_time_list, Gemma_2_9B_scores, Gemma_2_9B_bleu_results = create_scores('data/baseline_results/baseline_Gemma-2-9B_outputs.csv')
Mistral_7B_Instruct_v0_3_sts_scores, Mistral_7B_Instruct_v0_3_time_list, Mistral_7B_Instruct_v0_3_scores, Mistral_7B_Instruct_v0_3_bleu_results = create_scores('data/baseline_results/baseline_Mistral-7B-Instruct-v0.3_outputs.csv')
Qwen3_8B_sts_scores, Qwen3_8B_time_list, Qwen3_8B_scores, Qwen3_8B_bleu_results = create_scores('data/baseline_results/baseline_Qwen3-8B_outputs.csv')
"""

model_list = ["SmolLM3", "Gemma-2-9B","Mistral-7B-Instruct-v0.3","Llama-3.2-3B-Instruct" ]

for model_name in model_list:
  if model_name == "SmolLM3":
    SmolLM3_mds_scores, SmolLM3_length_scores, SmolLM3_sts_scores, SmolLM3_time_list, SmolLM3_scores, SmolLM3_bleu_results = create_scores_1(df_output, model_name)
    print(f"{model_name} done")
  elif model_name == "Gemma-2-9B":
    Gemma_2_9B_mds_scores, Gemma_2_9B_length_scores, Gemma_2_9B_sts_scores, Gemma_2_9B_time_list, Gemma_2_9B_scores, Gemma_2_9B_bleu_results = create_scores_1(df_output, model_name)
    print(f"{model_name} done")
  elif model_name == "Mistral-7B-Instruct-v0.3":
    Mistral_7B_Instruct_v0_3_mds_scores, Mistral_7B_Instruct_v0_3_length_scores, Mistral_7B_Instruct_v0_3_sts_scores, Mistral_7B_Instruct_v0_3_time_list, Mistral_7B_Instruct_v0_3_scores, Mistral_7B_Instruct_v0_3_bleu_results = create_scores_1(df_output, model_name)
    print(f"{model_name} done")
  elif model_name == "Llama-3.2-3B-Instruct":
    Llama_3_2_3B_Instruct_mds_scores, Llama_3_2_3B_Instruct_length_scores,Llama_3_2_3B_Instruct_sts_scores, Llama_3_2_3B_Instruct_time_list, Llama_3_2_3B_Instruct_scores, Llama_3_2_3B_Instruct_bleu_results = create_scores_1(df_output, model_name)
    print(f"{model_name} done")



Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

  source_framework mds_scores       rouge_scores       length_scores        \
                       median count       median count        median count   
0        ISO 27001   0.203933     6     0.149236     6      0.236594     6   
1         NIST CSF   0.216028    32     0.153014    32      0.256158    32   
2             SOC2   0.188252    27     0.144681    27      0.223350    27   

  sts_scores        
      median count  
0   0.779785     6  
1   0.802002    32  
2   0.783203    27  
SmolLM3 done


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

  source_framework mds_scores       rouge_scores       length_scores        \
                       median count       median count        median count   
0        ISO 27001   0.243858     6     0.193585     6      0.280230     6   
1         NIST CSF   0.250647    32     0.176670    32      0.293504    32   
2             SOC2   0.225780    27     0.189655    27      0.250000    27   

  sts_scores        
      median count  
0   0.771729     6  
1   0.797363    32  
2   0.787598    27  
Mistral-7B-Instruct-v0.3 done


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

  source_framework mds_scores       rouge_scores       length_scores        \
                       median count       median count        median count   
0        ISO 27001   0.221135     6     0.188804     6      0.222138     6   
1         NIST CSF   0.230151    32     0.174063    32      0.241809    32   
2             SOC2   0.188564    27     0.152091    27      0.212560    27   

  sts_scores        
      median count  
0   0.773682     6  
1   0.796631    32  
2   0.759766    27  
Llama-3.2-3B-Instruct done


In [ ]:
def _stack_offsets(
    x_vals: np.ndarray,
    *,
    base_y: float,
    step: float = 0.35,
    precision: int = 3,
) -> np.ndarray:
    """
    Deterministic "beeswarm-like" vertical offsets for repeated x values.
    - Quantizes x by rounding to `precision` decimals.
    - For each quantized bucket, assigns offsets symmetrically around base_y:
        0 -> 0
        1 -> +step
        2 -> -step
        3 -> +2*step
        4 -> -2*step
        ...
    Returns an array of y positions with same shape as x_vals.
    """
    x_vals = np.asarray(x_vals, dtype=float)
    q = np.round(x_vals, precision)
    y = np.empty_like(x_vals, dtype=float)

    # For reproducible ordering, we preserve the original order within each bucket
    # and assign offsets in a symmetric pattern.
    unique_buckets = np.unique(q)
    for b in unique_buckets:
        idx = np.where(q == b)[0]
        n = len(idx)
        if n == 1:
            y[idx[0]] = base_y
            continue

        # symmetric offsets: 0, +1, -1, +2, -2, ...
        ranks = np.arange(n)
        offsets = np.zeros(n, dtype=float)
        # build symmetric sequence
        seq = [0]
        k = 1
        while len(seq) < n:
            seq.append(+k)
            if len(seq) < n:
                seq.append(-k)
            k += 1
        offsets = np.array(seq, dtype=float) * step
        y[idx] = base_y + offsets

    return y


def plot_horizontal_violin_beeswarm_stacked(
    rouge_by_group: Dict[str, List[float]],
    *,
    ax: Optional[plt.Axes] = None,
    xlabel: str = "ROUGE-1",
    line_stat: str = "median",   # 'median' or 'mean'
    cap_height: float = 0.12,
    line_weight: float = 2.0,
    stack_step: float = 0.035,
    precision: int = 3,
    show_global_median: bool = True,
    max_possible: float = 1.0,
    min_possible: float = 0.0,
    title: str = "ROUGE-1 Scores by Model",
    label_suffix: str = "",
    base_color: str = "",   # distribution fill color (default blue)
    dot_color: str = "",
    hide_label: bool = False,
) -> Tuple[plt.Figure, plt.Axes]:
    """
    Multi-row horizontal violin + stacked dots + whisker line with end caps.
    Each group (key) gets its own row.

    Parameters are the same as before, plus:
      - ax: optional matplotlib Axes to draw into.
    Returns (fig, ax) without calling plt.show().
    """
    if not isinstance(rouge_by_group, dict) or not rouge_by_group:
        raise ValueError("rouge_by_group must be a non-empty dict of {label: list_of_scores}.")

    labels = list(rouge_by_group.keys())
    data = [np.asarray(rouge_by_group[k], dtype=float) for k in labels]
    positions = np.arange(1, len(labels) + 1, dtype=float)

    created_fig = ax is None
    if created_fig:
        fig, ax = plt.subplots(figsize=(9, max(2.0, 0.9 + 0.6 * len(labels))))
    else:
        fig = ax.figure


    ttl = title if not label_suffix else f"{title} — {label_suffix}"
    ax.set_title(ttl)

    # Shaded violins per row
    parts = ax.violinplot(
        data,
        positions=positions,
        vert=False,
        showmeans=False,
        showextrema=False,
        showmedians=False,
    )
    # Light fill for violins
    for pc in parts["bodies"]:
        if base_color != "":
            pc.set_facecolor(base_color)
            pc.set_edgecolor(base_color)
        pc.set_alpha(0.2)
        pc.set_linewidth(0.8)


    # Per-row overlays
    for pos, vals in zip(positions, data):
        if vals.size == 0:
            continue

        # stacked dots (deterministic)
        y = _stack_offsets(vals, base_y=pos, step=stack_step, precision=precision)

        if dot_color != "":
            ax.scatter(vals, y, s=14, alpha=0.85, color=dot_color)
        else:
            ax.scatter(vals, y, s=14, alpha=0.85)

        # whisker line min→max + end caps
        vmin, vmax = float(np.min(vals)), float(np.max(vals))
        ax.hlines(pos, vmin, vmax, linewidth=line_weight)
        # faint extension to max_possible (if vmax < max_possible)
        if vmin > min_possible:
            ax.hlines(
                    pos,
                    min_possible,
                    vmin,
                    linewidth=line_weight,
                    color="gray",
                    alpha=0.3,
                    linestyles="dashed",
                )
        if vmax < max_possible:
            ax.hlines(
                    pos,
                    vmax,
                    max_possible,
                    linewidth=line_weight,
                    color="gray",
                    alpha=0.3,
                    linestyles="dashed"
                )

        cap_y1, cap_y2 = pos - cap_height / 2.0, pos + cap_height / 2.0
        ax.vlines([vmin, vmax], cap_y1, cap_y2, linewidth=line_weight)

        # center stat marker (median or mean)
        stat_val = float(np.median(vals) if line_stat == "median" else np.mean(vals))
        ax.vlines(stat_val, cap_y1, cap_y2, linewidth=line_weight)

    # Optional global median line across all rows
    if show_global_median:
        all_vals = np.concatenate([v for v in data if v.size])
        if all_vals.size:
            stat_val = float(np.median(all_vals) if line_stat == "median" else np.mean(all_vals))
            ax.axvline(stat_val, color="darkgray", linestyle="-", linewidth=line_weight,
                       label=f"{line_stat.title()} = {stat_val:.3f}")

            # Text label near the top, like BLEU avg
            # ax.text(stat_val, len(labels) + 0.6, f"{line_stat.title()} = {stat_val:.3f}",
            #         color="gray", fontsize=9, ha="center", va="bottom")


            ax.legend(frameon=False, fontsize=9, loc="upper right")


    # Styling like a SHAP beeswarm
    ax.set_yticks(positions)

    ax.set_yticklabels(labels)
    for label_text, tick in zip(labels, ax.yaxis.get_major_ticks()):
        if not label_text:  # catches '' or None
            tick.tick1line.set_visible(False)  # left tick
            tick.tick2line.set_visible(False)  # right tick
            tick.label1.set_visible(False)     # hide text as well just in case
    # if xlabel != "":
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", linestyle=":", alpha=0.5)
    ax.set_ylim(0.5, len(labels) + 0.5)
    # remove all borders
    for spine in ax.spines.values():
        spine.set_visible(False)


    if created_fig:
        fig.tight_layout()
    return fig, ax


In [ ]:
def _fmt(x: float) -> str:
    if x >= 0.1:   return f"{x:.2f}"
    if x >= 0.01:  return f"{x:.3f}"
    return f"{x:.4f}"


def _apply_clean_axes(ax):
    ax.grid(True, axis="y", linestyle="--", linewidth=0.6, alpha=0.6)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)


def plot_single_model_bleu_precisions(
    bleu_results: Dict[str, Any],
    *,
    ax: Optional[plt.Axes] = None,
    ylim: Tuple[float, float] = (0.0, 1.0),
    annotate: bool = True,
    rotation: int = 0,
    title: str = "BLEU n-gram Precisions (single model)",
    ylabel: str = "Precision (0–1)",
    label_suffix: Optional[str] = None,
) -> Tuple[plt.Figure, plt.Axes]:
    """
    Plot 1–4 gram precisions with colored bars and a BLEU average line.
    Expects: bleu_results["precisions"] (list of length 4) and bleu_results["bleu"] (float).
    """
    precisions: Sequence[float] = bleu_results.get("precisions", [float("nan")] * 4)
    p = (list(precisions) + [float("nan")] * 4)[:4]

    created_fig = ax is None
    if created_fig:
        fig, ax = plt.subplots(figsize=(6, 5))
    else:
        fig = ax.figure

    # harmonious palette (you can swap or override if desired)
    colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

    x = np.arange(1, 5)
    bars = ax.bar(x, p, color=colors[: len(p)], edgecolor="black", linewidth=0.5)

    # add BLEU average line if available
    bleu_score = bleu_results.get("bleu")
    if bleu_score is not None and not math.isnan(bleu_score):
        ax.axhline(bleu_score, color="darkgray", linestyle="-", linewidth=1,
                   label=f"BLEU avg = {_fmt(bleu_score)}")
        ax.legend(frameon=False, fontsize=9, loc="upper right")

    # annotate bars
    if annotate:
        for rect, val in zip(bars, p):
            if not math.isnan(val):
                ax.text(rect.get_x() + rect.get_width()/2.0, rect.get_height(),
                        _fmt(val), ha="center", va="bottom", fontsize=9)

    # labels & formatting
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel)
    ttl = title if not label_suffix else f"{title} — {label_suffix}"
    ax.set_title(ttl)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{i}-gram" for i in x], rotation=rotation, ha="center")

    _apply_clean_axes(ax)

    # remove all borders
    for spine in ax.spines.values():
        spine.set_visible(False)


    # footer info
    bp = bleu_results.get("brevity_penalty", None)
    lr = bleu_results.get("length_ratio", None)
    footer_bits = []
    if bp is not None: footer_bits.append(f"BP={_fmt(float(bp))}")
    if lr is not None: footer_bits.append(f"LenRatio={_fmt(float(lr))}")
    if footer_bits:
        ax.text(0.98, -0.12, "  •  ".join(footer_bits), transform=ax.transAxes,
                ha="right", va="top", fontsize=9)

    if created_fig:
        fig.tight_layout()
    return fig, ax


In [ ]:
def set_plot_font_styles(
    fig,
    axis_size: int = 10,
    title_size: int = 12,
    annotation_size: int = 9,
    title_pad: int = 10,
) -> None:
    """
    Apply consistent font sizes and title spacing to all Axes in a matplotlib Figure.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Figure containing one or more Axes.
    axis_size : int
        Font size for axis labels and tick labels.
    title_size : int
        Font size for subplot titles.
    annotation_size : int
        Font size for legends and annotation text.
    title_pad : int
        Padding (points) between title and plot area.
    """
    for ax in fig.get_axes():
        # --- Title ---
        ax.title.set_fontsize(title_size)
        # Set consistent title padding (distance above axes)
        title_text = ax.get_title()
        if title_text:
            ax.set_title(title_text, fontsize=title_size, pad=title_pad)

        # --- Axis labels ---
        ax.xaxis.label.set_fontsize(axis_size)
        ax.yaxis.label.set_fontsize(axis_size)

        # --- Tick labels ---
        for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
            tick_label.set_fontsize(axis_size)

        # --- Legend text ---
        leg = ax.get_legend()
        if leg is not None:
            leg.get_title().set_fontsize(annotation_size)
            for text in leg.get_texts():
                text.set_fontsize(annotation_size)

        # --- Annotation text (e.g., BLEU avg or Median labels) ---
        for child in ax.get_children():
            if isinstance(child, plt.Text):
                txt = child.get_text()
                # Skip axis labels & titles
                if txt not in {ax.get_title(), ax.get_xlabel(), ax.get_ylabel()}:
                    child.set_fontsize(annotation_size)

    # Redraw to apply all sizing and layout
    fig.canvas.draw_idle()


## Rogues/BLEU scores for All Models

In [ ]:
fig, axes = plt.subplots(
    4, 2,
    figsize=(12, 12),
    gridspec_kw={"width_ratios": [7, 3]}   # 70/30 width split
)
axes = axes.flatten()

plot_single_model_bleu_precisions(SmolLM3_bleu_results, ax=axes[1], label_suffix="", title= "BLEU n-gram Precisions (SmolLM3)")
plot_single_model_bleu_precisions(Gemma_2_9B_bleu_results, ax=axes[3], label_suffix="", title= "BLEU n-gram Precisions (Gemma-2-9B)")
plot_single_model_bleu_precisions(Llama_3_2_3B_Instruct_bleu_results, ax=axes[5], label_suffix="", title= "BLEU n-gram Precisions (Llama-3.2-3B-Instruct)")
plot_single_model_bleu_precisions(Mistral_7B_Instruct_v0_3_bleu_results, ax=axes[7], label_suffix="", title= "BLEU n-gram Precisions (Mistral-7B-Instruct-v0.3)")
#plot_single_model_bleu_precisions(Qwen3_8B_bleu_results, ax=axes[9], label_suffix="", title= "BLEU n-gram Precisions (Qwen3-8B)")

plot_horizontal_violin_beeswarm_stacked(SmolLM3_scores, ax=axes[0], xlabel="", title= "ROUGE Scores (SmolLM3)",cap_height=0.5, line_weight=1, base_color="#222222")
plot_horizontal_violin_beeswarm_stacked(Gemma_2_9B_scores, ax=axes[2], xlabel="", title= "ROUGE Scores (Gemma-2-9B)",cap_height=0.5, line_weight=1, base_color="#222222")
plot_horizontal_violin_beeswarm_stacked(Llama_3_2_3B_Instruct_scores, ax=axes[4], xlabel="", title= "ROUGE Scores (Llama-3.2-3B-Instruct)",cap_height=0.5, line_weight=1, base_color="#222222")
plot_horizontal_violin_beeswarm_stacked(Mistral_7B_Instruct_v0_3_scores, ax=axes[6], xlabel="", title= "ROUGE Scores (Mistral-7B-Instruct-v0.3)",cap_height=0.5, line_weight=1, base_color="#222222")
#plot_horizontal_violin_beeswarm_stacked(Qwen3_8B_scores, ax=axes[8], xlabel="", title= "ROUGE Scores (Qwen3-8B)",cap_height=0.5, line_weight=1, base_color="#222222")




fig.subplots_adjust(hspace=0.6)
set_plot_font_styles(fig, axis_size=8, title_size=10, annotation_size=7)
plt.show()

## LLM-STS (BAAI/bge-large-en-v1.5) scores for All Models

In [ ]:
fig, axes = plt.subplots(
    4, 1,
    figsize=(12, 10),
    # gridspec_kw={"width_ratios": [7, 3]}   # 70/30 width split
)
axes = axes.flatten()
plot_horizontal_violin_beeswarm_stacked({None: SmolLM3_sts_scores}, ax=axes[0]
                                        , xlabel="Cosine Similarity"
                                        , line_stat="median"
                                        , title= "CosSim Scores (LLM-STS) - SmolLM3"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#d62728"
                                        # , hide_label=True
                                        )

plot_horizontal_violin_beeswarm_stacked({"": Gemma_2_9B_sts_scores}, ax=axes[1]
                                        , xlabel="Cosine Similarity"
                                        , line_stat="median"
                                        , title= "CosSim Scores (LLM-STS) - Gemma_2_9B"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#2ca02c"
                                        , hide_label=True
                                        )

plot_horizontal_violin_beeswarm_stacked({"": Llama_3_2_3B_Instruct_sts_scores}, ax=axes[2]
                                        , xlabel="Cosine Similarity"
                                        , line_stat="median"
                                        , title= "CosSim Scores (LLM-STS) - Llama-3.2-3B-Instruct"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#ff7f0e"
                                        , hide_label=True
                                        )
plot_horizontal_violin_beeswarm_stacked({"": Mistral_7B_Instruct_v0_3_sts_scores}, ax=axes[3]
                                        , xlabel="Cosine Similarity"
                                        , line_stat="median"
                                        , title= "CosSim Scores (LLM-STS) - Mistral-7B-Instruct-v0.3"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#1f77b4"
                                        , hide_label=True
                                        )
"""
plot_horizontal_violin_beeswarm_stacked({"": Qwen3_8B_sts_scores}, ax=axes[4]
                                        , xlabel="Cosine Similarity"
                                        , line_stat="median"
                                        , title= "CosSim Scores (LLM-STS) - Qwen3-8B"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#9467bd"
                                        , hide_label=True
                                        )
"""

# plot_horizontal_violin_beeswarm_stacked({"Llama-3.2-3B-Instruct": Llama_3_2_3B_Instruct_sts_scores}, ax=axes[1], xlabel="", title= "ROUGE Scores (Llama-3.2-3B-Instruct)",cap_height=0.5, line_weight=1)
# fig.tight_layout()
fig.subplots_adjust(hspace=0.6)
set_plot_font_styles(fig, axis_size=8, title_size=10, annotation_size=7)
plt.show()


## Length score for all Models

In [ ]:
fig, axes = plt.subplots(
    4, 1,
    figsize=(12, 10),
    # gridspec_kw={"width_ratios": [7, 3]}   # 70/30 width split
)
axes = axes.flatten()
plot_horizontal_violin_beeswarm_stacked({None: SmolLM3_length_scores}, ax=axes[0]
                                        , xlabel="Length Similarity"
                                        , line_stat="median"
                                        , title= "Length Scores - SmolLM3"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#d62728"
                                        # , hide_label=True
                                        )

plot_horizontal_violin_beeswarm_stacked({"": Gemma_2_9B_mds_scores}, ax=axes[1]
                                        , xlabel="Length Similarity"
                                        , line_stat="median"
                                        , title= "Length Score - Gemma_2_9B"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#2ca02c"
                                        , hide_label=True
                                        )

plot_horizontal_violin_beeswarm_stacked({"": Llama_3_2_3B_Instruct_mds_scores}, ax=axes[2]
                                        , xlabel="Length Similarity"
                                        , line_stat="median"
                                        , title= "Length Score  - Llama-3.2-3B-Instruct"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#ff7f0e"
                                        , hide_label=True
                                        )
plot_horizontal_violin_beeswarm_stacked({"": Mistral_7B_Instruct_v0_3_mds_scores}, ax=axes[3]
                                        , xlabel="Length Similarity"
                                        , line_stat="median"
                                        , title= "Length Scores - Mistral-7B-Instruct-v0.3"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#1f77b4"
                                        , hide_label=True
                                        )
"""
plot_horizontal_violin_beeswarm_stacked({"": Qwen3_8B_sts_scores}, ax=axes[4]
                                        , xlabel="Cosine Similarity"
                                        , line_stat="median"
                                        , title= "CosSim Scores (LLM-STS) - Qwen3-8B"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#9467bd"
                                        , hide_label=True
                                        )
"""

# plot_horizontal_violin_beeswarm_stacked({"Llama-3.2-3B-Instruct": Llama_3_2_3B_Instruct_sts_scores}, ax=axes[1], xlabel="", title= "ROUGE Scores (Llama-3.2-3B-Instruct)",cap_height=0.5, line_weight=1)
# fig.tight_layout()
fig.subplots_adjust(hspace=0.6)
set_plot_font_styles(fig, axis_size=8, title_size=10, annotation_size=7)
plt.show()

## MDS scores for All Models

In [ ]:
fig, axes = plt.subplots(
    4, 1,
    figsize=(12, 10),
    # gridspec_kw={"width_ratios": [7, 3]}   # 70/30 width split
)
axes = axes.flatten()
plot_horizontal_violin_beeswarm_stacked({None: SmolLM3_mds_scores}, ax=axes[0]
                                        , xlabel="Multi Dimension Similarity"
                                        , line_stat="median"
                                        , title= "MDS Scores - SmolLM3"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#d62728"
                                        # , hide_label=True
                                        )

plot_horizontal_violin_beeswarm_stacked({"": Gemma_2_9B_mds_scores}, ax=axes[1]
                                        , xlabel="Multi Dimension Similarity"
                                        , line_stat="median"
                                        , title= "MDS Scores - Gemma_2_9B"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#2ca02c"
                                        , hide_label=True
                                        )

plot_horizontal_violin_beeswarm_stacked({"": Llama_3_2_3B_Instruct_mds_scores}, ax=axes[2]
                                        , xlabel="Multi Dimension Similarity"
                                        , line_stat="median"
                                        , title= "MDS Scores  - Llama-3.2-3B-Instruct"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#ff7f0e"
                                        , hide_label=True
                                        )
plot_horizontal_violin_beeswarm_stacked({"": Mistral_7B_Instruct_v0_3_mds_scores}, ax=axes[3]
                                        , xlabel="Multi Dimension Similarity"
                                        , line_stat="median"
                                        , title= "MDS Scores - Mistral-7B-Instruct-v0.3"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#1f77b4"
                                        , hide_label=True
                                        )
"""
plot_horizontal_violin_beeswarm_stacked({"": Qwen3_8B_sts_scores}, ax=axes[4]
                                        , xlabel="Cosine Similarity"
                                        , line_stat="median"
                                        , title= "CosSim Scores (LLM-STS) - Qwen3-8B"
                                        , max_possible=1.0
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#9467bd"
                                        , hide_label=True
                                        )
"""

# plot_horizontal_violin_beeswarm_stacked({"Llama-3.2-3B-Instruct": Llama_3_2_3B_Instruct_sts_scores}, ax=axes[1], xlabel="", title= "ROUGE Scores (Llama-3.2-3B-Instruct)",cap_height=0.5, line_weight=1)
# fig.tight_layout()
fig.subplots_adjust(hspace=0.6)
set_plot_font_styles(fig, axis_size=8, title_size=10, annotation_size=7)
plt.show()

## Execution Time Distriobution (local Quant 4K_M Files)

In [ ]:

fig, axes = plt.subplots(
    4, 1,
    figsize=(12, 10),
    # gridspec_kw={"width_ratios": [7, 3]}   # 70/30 width split
)
axes = axes.flatten()
max_time = max(SmolLM3_time_list + Gemma_2_9B_time_list + Llama_3_2_3B_Instruct_time_list + Mistral_7B_Instruct_v0_3_time_list ) #+ Qwen3_8B_time_list
max_time = math.ceil(max_time * 1.1)
plot_horizontal_violin_beeswarm_stacked({None: SmolLM3_time_list}, ax=axes[0]
                                        , xlabel="Execution Time (seconds)"
                                        , line_stat="median"
                                        , title= "Execution Time (seconds) - SmolLM3"
                                        , max_possible=max_time
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#d62728"
                                        , stack_step = 0.055
                                        , precision = 1,
                                        # , hide_label=True
                                        )

plot_horizontal_violin_beeswarm_stacked({None: Gemma_2_9B_time_list}, ax=axes[1]
                                        , xlabel="Execution Time (seconds)"
                                        , line_stat="median"
                                        , title= "Execution Time (seconds) - Gemma-2-9B"
                                        , max_possible=max_time
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#2ca02c"
                                        , stack_step = 0.055
                                        , precision = 1,
                                        # , hide_label=True
                                        )
plot_horizontal_violin_beeswarm_stacked({None: Llama_3_2_3B_Instruct_time_list}, ax=axes[2]
                                        , xlabel="Execution Time (seconds)"
                                        , line_stat="median"
                                        , title= "Execution Time (seconds) - Llama-3.2-3B-Instruct"
                                        , max_possible=max_time
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#ff7f0e"
                                        , stack_step = 0.055
                                        , precision = 1,
                                        # , hide_label=True
                                        )
plot_horizontal_violin_beeswarm_stacked({None: Mistral_7B_Instruct_v0_3_time_list}, ax=axes[3]
                                        , xlabel="Execution Time (seconds)"
                                        , line_stat="median"
                                        , title= "Execution Time (seconds) - Mistral-7B-Instruct-v0.3"
                                        , max_possible=max_time
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#1f77b4"
                                        , stack_step = 0.055
                                        , precision = 1,
                                        # , hide_label=True
                                        )
"""
plot_horizontal_violin_beeswarm_stacked({None: Qwen3_8B_time_list}, ax=axes[4]
                                        , xlabel="Execution Time (seconds)"
                                        , line_stat="median"
                                        , title= "Execution Time (seconds) - Qwen3-8B"
                                        , max_possible=max_time
                                        , cap_height=0.5
                                        , line_weight=1
                                        , base_color="#222222"
                                        , dot_color="#9467bd"
                                        , stack_step = 0.055
                                        , precision = 1,
                                        # , hide_label=True
                                        )
  """

fig.subplots_adjust(hspace=0.6)
set_plot_font_styles(fig, axis_size=8, title_size=10, annotation_size=7)
plt.show()
